# 02b · Evaluating the merge

Scores one saved merge run (`runs/merge/`, made by notebook 02) against the hand-labelled step-2 golden set
(`eval/golden/merging_and_summarizing.yaml`): every unique risk of the report once, with all its locations.

- **How:** one LLM judge call gets the golden and the predicted records, **pairs** the ones that describe the same
  risk, and grades each pair's title, description and mitigation `yes` or `no` (`merge-judge`). Code checks
  the pairs and grades what needs no judge: category, and whether the right mentions were merged (pages).
- **Code checks without a golden:** the rules every record must follow (description at most 3 sentences,
  mitigation "None" exactly when no mention states one, a short title).
- **What we report:** recall, precision and F1 over records, the field grades, and the checks. Recall leads
  (plan: a missed principal risk costs more than an extra one).

**What this measures.** The merge run is built on a step-1 run, so a risk step 1 missed is missed here too. The
scores are end to end (steps 1 + 2); the misses list says where each miss came from.

The code lives in `annual_report_reader/evaluation/`, one module per section (named in its heading). It was
written here first and then lifted; the modules are unit-tested (`tests/evaluation/test_merge_evaluation.py`).

## 1. Config → `evaluation/config.py`

The same `JudgeConfig` as step 1, with the step-2 golden set and judge prompt. The judge is Sol, not the model
that merged. The merge run to score is named explicitly.

In [1]:
from pathlib import Path

from dotenv import load_dotenv

from annual_report_reader.evaluation.config import JudgeConfig
from annual_report_reader.extraction.config import REPO_ROOT

load_dotenv(REPO_ROOT / ".env")  # endpoint, key and deployment names

MERGE_RUN = "merge-20261112T012912Z-53ad"  # merge on the Luna high-effort extraction run
JUDGE_CONFIG = JudgeConfig(
    golden_path=Path("eval/golden/merging_and_summarizing.yaml"),
    judge_prompt="merge-judge",
)
JUDGE_CONFIG

JudgeConfig(golden_path=PosixPath('eval/golden/merging_and_summarizing.yaml'), judge_deployment='gpt-5.6-sol-carl-test', judge_effort='medium', judge_prompt_dir=PosixPath('prompts'), judge_prompt='merge-judge', max_concurrency=4, price_in_per_m=None, price_out_per_m=None)

## 2. Golden set → `evaluation/golden.py`

Each unique risk once, with every location (most informative first). `secondary` in the file is
`secondary_category` in code, as in step 1. Unknown keys are rejected, so a typo fails on load.

In [2]:
from annual_report_reader.evaluation.golden import load_golden_records

golden = load_golden_records(JUDGE_CONFIG.resolve(JUDGE_CONFIG.golden_path))
print(f"{len(golden)} golden records")
for i, g in enumerate(golden):
    print(f"  g{i:<2} {g.title} (p.{', '.join(str(loc.page) for loc in g.locations)})")

11 golden records
  g0  Geopolitical conflict and trade barriers (p.51)
  g1  Shifting industrial policy and regulation (p.51)
  g2  Project execution (p.51)
  g3  Cyber security risks (p.118, 51, 74)
  g4  Cyber security incidents (p.118)
  g5  Carbon taxes and tariffs (p.85, 71)
  g6  Cost implication of injuries in own workforce (p.73)
  g7  Cost implications of injuries to contractors and sub-contractors (p.73)
  g8  Fines related to forced or child labour (p.73)
  g9  Insufficient market conditions (p.74)
  g10 Risk of corruption and bribery (p.74)


## 3. The merge run → `merge/runs.py`

The saved run holds everything needed: the records, and the mentions they were built from. A run whose call failed
has no records; it is still scored, and every golden risk counts as missed, because that is what the analyst
would get.

In [3]:
from annual_report_reader.merge.runs import MERGE_RUNS_DIR, load_merge_run

merge_run = load_merge_run(REPO_ROOT / MERGE_RUNS_DIR / f"{MERGE_RUN}.json")
print(
    f"{merge_run.run_id}: {len(merge_run.records)} records from {len(merge_run.mentions)} mentions, "
    f"extraction run {merge_run.config.extraction_run}, prompt {merge_run.config.prompt}"
    + (f" | MERGE CALL FAILED: {merge_run.call.error}" if merge_run.call.error else "")
)
for r in merge_run.records:
    print(f"  {r.id} {r.title} (p.{', '.join(str(loc.page) for loc in r.locations)})")

merge-20261112T012912Z-53ad: 9 records from 12 mentions, extraction run extract-20261112T000123Z, prompt merge
  r00 Geopolitics and regulatory framework (p.51)
  r01 Project execution (p.51)
  r02 Cyber security risks (p.51, 118, 74)
  r03 Carbon taxes and tariffs (p.85, 71)
  r04 Injuries in the own workforce (p.73)
  r05 Injuries to value chain workers (p.73)
  r06 Forced or child labour fines (p.73)
  r07 Insufficient market conditions (p.74)
  r08 Corruption and bribery (p.74)


## 4. Judge → `evaluation/merge_judge.py`

One call for the whole report: the judge needs every record in view to decide what pairs with what. It gets both
lists as JSON, each record with an integer id, its title, description, mitigation and pages. Category is left out
(code grades it). The prompt is a file (`prompts/merge-judge.md`), sent unchanged.

Grades are `yes`/`no` only, as in step 1. The rules for the two merge mistakes:

- a prediction that **combines** two golden risks pairs with one of them; the other is a miss;
- two predictions of the **same** golden risk: one pairs, the other is an extra.

In [4]:
from annual_report_reader.evaluation.judge import load_judge_prompt
from annual_report_reader.evaluation.merge_judge import merge_payload

judge_prompt = load_judge_prompt(JUDGE_CONFIG.resolve(JUDGE_CONFIG.judge_prompt_dir), JUDGE_CONFIG.judge_prompt)
print(merge_payload(golden, merge_run.records)[:1200], "...")

{
 "golden": [
  {
   "id": 0,
   "title": "Geopolitical conflict and trade barriers",
   "description": "Conflicts in Ukraine and the Middle East and intensifying trade tensions, with tariffs, sanctions, and export controls on e.g. critical materials, are disrupting global supply chains and business operations. This can delay deliveries and raise Vestas’ costs through tariffs and trade barriers, challenging cost efficiency.",
   "mitigation": "We manage geopolitical risks by continuously monitoring global developments and implementing appropriate mitigations. By maintaining a global and regionalised manufacturing footprint, Vestas can effectively respond to geopolitical shifts and regulatory changes. Additionally, we exercise commercial discipline and utilise indexation and financial hedging strategies to minimise exposure.",
   "pages": [
    51
   ]
  },
  {
   "id": 1,
   "title": "Shifting industrial policy and regulation",
   "description": "Governments in the EU, China, and the 

Run the judge. Like every call in this project, a failure comes back as data (`judge_error`), not as an
exception, and nothing is reused from earlier calls.

In [5]:
import time
from datetime import UTC, datetime

from annual_report_reader.chat import azure_chat_model
from annual_report_reader.evaluation.judge import eval_id_for
from annual_report_reader.evaluation.merge_judge import judge_merge, merge_judge_chain

judge = merge_judge_chain(judge_prompt, azure_chat_model(JUDGE_CONFIG.judge_deployment, JUDGE_CONFIG.judge_effort))

started_at = datetime.now(UTC)
eval_id = eval_id_for(merge_run.run_id, JUDGE_CONFIG.judge_prompt, started_at)
start = time.perf_counter()
result = judge_merge(judge, golden, merge_run.records, JUDGE_CONFIG, eval_id)
elapsed = time.perf_counter() - start
print(
    f"{eval_id}: {elapsed:.0f}s | {result.usage}"
    + (f" | JUDGE FAILED: {result.judge_error}" if result.judge_error else "")
)

merge-20261112T012912Z-53ad--merge-judge--20261112T012929Z-b1fe: 7s | input_tokens=2827 output_tokens=709 reasoning_tokens=150


## 5. Pairs → `evaluation/merge_judge.py`

As in step 1, the pairs must be one-to-one with valid ids, which code checks: the first valid pair wins, any other
pair is **dropped with its reason** and kept. Each kept pair also gets the code's grades:

- `category`: the predicted primary category is the golden primary; `category_any`: it is the golden primary or
  secondary;
- `pages`: the record's pages are exactly the golden pages, i.e. the right mentions were merged;
- `first_page`: the first location is the golden first one, i.e. the most informative mention was picked.

A pair whose description is graded `no` stays a pair: it counts as found, with a bad description, as in step 1.
Misses and extras are listed with a hint: a missed golden risk on a page where step 1 found nothing is a step-1
miss; a missed golden risk whose pages overlap a paired record may have been merged into it.

In [6]:
import pandas as pd

for d in result.dropped:
    print(f"dropped g{d.verdict.golden_id} ↔ r{d.verdict.pred_id}: {d.why}")

pd.set_option("display.max_colwidth", None)
pd.DataFrame(
    [
        {"golden": golden[p.golden_id].title, "predicted": merge_run.records[p.pred_id].title}
        | p.model_dump(exclude={"golden_id", "pred_id", "reason"})
        for p in result.pairs
    ]
)

,golden,predicted,title,description,mitigation,category,category_any,pages,first_page
0,Geopolitical conflict and trade barriers,Geopolitics and regulatory framework,yes,yes,yes,no,no,yes,yes
1,Project execution,Project execution,yes,yes,yes,yes,yes,yes,yes
2,Cyber security risks,Cyber security risks,yes,yes,no,yes,yes,yes,no
3,Carbon taxes and tariffs,Carbon taxes and tariffs,yes,yes,yes,no,no,yes,yes
4,Cost implication of injuries in own workforce,Injuries in the own workforce,yes,yes,yes,no,no,yes,yes
5,Cost implications of injuries to contractors and sub-contractors,Injuries to value chain workers,yes,yes,yes,no,no,yes,yes
6,Fines related to forced or child labour,Forced or child labour fines,yes,yes,yes,no,no,yes,yes
7,Insufficient market conditions,Insufficient market conditions,yes,yes,yes,yes,yes,yes,yes
8,Risk of corruption and bribery,Corruption and bribery,yes,yes,yes,yes,yes,yes,yes


Misses and extras, each with where it most likely came from.

In [7]:
from annual_report_reader.evaluation.merge_judge import miss_hint

paired_g = {p.golden_id for p in result.pairs}
paired_p = {p.pred_id for p in result.pairs}
print(f"Missed golden risks ({len(golden) - len(paired_g)}):")
for i, g in enumerate(golden):
    if i not in paired_g:
        print(f"  g{i} {g.title}: {miss_hint(g, merge_run.records, result.pairs, merge_run.mentions)}")
print(f"Extra records ({len(merge_run.records) - len(paired_p)}):")
for i, r in enumerate(merge_run.records):
    if i not in paired_p:
        print(f"  {r.id} {r.title} (p.{', '.join(str(loc.page) for loc in r.locations)})")

Missed golden risks (2):
  g1 Shifting industrial policy and regulation: shares pages with r00, r01, r02: merged into it, or missed by step 1 on that page
  g4 Cyber security incidents: shares pages with r02: merged into it, or missed by step 1 on that page
Extra records (0):


## 6. Code checks → `evaluation/merge_checks.py`

Rules every record must follow, checked without a golden or a judge:

- the description has **at most 3 sentences** (the prompt asks for 2, at most 3);
- the mitigation is `"None"` **exactly when** none of the record's mentions states one, so the merge neither
  invented nor dropped a mitigation;
- the title has **at most 10 words** (the prompt asks for about 8).

Sentences are counted by splitting on `.`, `!` or `?` followed by a space and a capital letter, so "e.g." and
"6.39" do not split. A record that was left out by the model and copied from step 1 can fail the first check: that
is expected, and visible.

In [8]:
from annual_report_reader.evaluation.merge_checks import check_record

checks = [check_record(r, merge_run.mentions) for r in merge_run.records]
print(f"{sum(c.passed for c in checks)} of {len(checks)} records pass every check")
pd.DataFrame([c.model_dump() for c in checks])

9 of 9 records pass every check


,record_id,sentences,description_max_3_sentences,mitigation_consistent,title_max_10_words,passed
0,r00,2,True,True,True,True
1,r01,2,True,True,True,True
2,r02,2,True,True,True,True
3,r03,2,True,True,True,True
4,r04,1,True,True,True,True
5,r05,1,True,True,True,True
6,r06,1,True,True,True,True
7,r07,1,True,True,True,True
8,r08,1,True,True,True,True


## 7. Scores → `evaluation/scores.py`

- **recall** = paired / golden, **precision** = paired / predicted, **F1** combines both (step 1's `Counts`);
- per field: the share of pairs graded `yes`;
- checks: the share of records that pass each check.

With one judge call per report there is no per-page breakdown; the misses list above says where each miss came
from.

In [9]:
from annual_report_reader.evaluation.merge_checks import CHECKS
from annual_report_reader.evaluation.scores import Counts, share_yes

counts = Counts(golden=len(golden), predicted=len(merge_run.records), paired=len(result.pairs))
if result.judge_error:
    print("NOT SCORED: the judge failed")
else:
    print(
        f"recall {counts.recall:.2f} | precision {counts.precision:.2f} | F1 {counts.f1:.2f}  "
        f"({counts.paired} of {counts.golden} golden found, {counts.predicted - counts.paired} extra)"
    )
fields = ["title", "description", "mitigation", "category", "category_any", "pages", "first_page"]
pd.Series(share_yes(result.pairs, fields) | share_yes(checks, CHECKS), name="share yes / passing").round(2).to_frame()

recall 0.82 | precision 1.00 | F1 0.90  (9 of 11 golden found, 0 extra)


,share yes / passing
title,1.00
description,1.00
mitigation,0.89
category,0.44
category_any,0.44
pages,1.00
first_page,0.89
description_max_3_sentences,1.00
mitigation_consistent,1.00
title_max_10_words,1.00


## 8. Save → `evaluation/runs.py`

Saved to `eval/runs/<merge run>--<judge prompt>--<time>.json`: the judge config, the judge prompt as sent, the
judge result (pairs, dropped pairs, usage), the checks and the counts. Every evaluation is kept.

In [10]:
from annual_report_reader.evaluation.runs import EVAL_RUNS_DIR, MergeEvaluationRun, load_merge_eval, save_merge_eval

evaluation = MergeEvaluationRun(
    eval_id=eval_id,
    started_at=started_at,
    elapsed_s=round(elapsed, 1),
    config=JUDGE_CONFIG,
    merge_run_id=merge_run.run_id,
    judge_system_prompt=judge_prompt.format_messages(payload="")[0].content,
    result=result,
    checks=checks,
    counts=counts,
)
eval_path = save_merge_eval(evaluation, REPO_ROOT / EVAL_RUNS_DIR)
assert load_merge_eval(eval_path) == evaluation  # the file loads back
print(f"Saved {eval_path.relative_to(REPO_ROOT)}")

Saved eval/runs/merge-20261112T012912Z-53ad--merge-judge--20261112T012929Z-b1fe.json


## 9. Observations

_To fill in after reviewing the scores._